# 06 · Advanced: writing the action by hand

The SDE builder writes the MSRJD action for you. Underneath, every model is a `TemporalModelBuilder` (`api.model`): fields, parameters, kernels, an action text and the mean-field equations. Writing that by hand is useful when

* a term is easier to state in the action than as an equation (for example a non-local noise kernel or a cumulant generating function that is not a source in the library),
* you start from a published action,
* you want to see exactly what the SDE builder produced.

Conventions: the response field of `x` is `xt`; the action is
$S = \int \mathrm dt\,\big[\tilde x\,(\dot x - f(x)) - K(\tilde x)\big]$ with $K$ the noise CGF (Itô); a Gaussian source of variance rate $2D$ contributes $-D\tilde x^2$. `Dt` is the time derivative.

Run time: under a minute.

In [1]:
%matplotlib inline
try:
    preparser(False)   # Sage kernel: plain Python syntax and numbers in the cells below
except NameError:
    pass
import os, sys, time, warnings
import numpy as np
import matplotlib.pyplot as plt
# locate the repository root (the folder holding the 'api' package) from any working directory
_root = os.path.abspath('')
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, 'api')):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)
os.chdir(_root)
def ells(r):   # C(tau) per loop order (real parts), a list indexed by the loop order
    return [np.real(np.asarray(r['C_tau_by_ell'][l])) for l in sorted(r['C_tau_by_ell'])]
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

In [2]:
from api.sde import SDE
from api.noise import Gaussian
from api.model import TemporalModelBuilder
from api import compute_cumulants
import daedalus as dd

## 1. The SDE builder shows its action

`show()` prints the builder calls of the equivalent hand-written model; `to_builder()` returns that builder.

In [3]:
sde = (SDE('quartic oscillator')
       .parameter('mu', default=1.0, domain='positive').parameter('eps', default=0.05, domain='positive')
       .parameter('D', default=0.5, domain='positive')
       .field('x').drift('x', '-mu*x - eps*x^3')
       .noise('xi', Gaussian(var='2*D'), couples={'x': 1}))
sde.show();

# SDE 'quartic oscillator' (ito)
.set_action_text('''
sum(
    xt[i]*(Dt*x[i] - (-mu*x[i] - eps*x[i]^3))
    - (D*xt[i]^2)
    for i in pop)
''')
.equation(lhs='Dt*x[i] + mu*x[i]', rhs='-eps*x[i]^3', population='pop')
# note: mean-field equation of 'x': linear term -mu*x[i] moved to the lhs.


## 2. The same model by hand

In [4]:
hand = (TemporalModelBuilder('quartic oscillator (by hand)')
        .physical_field('x')
        .parameter('mu', default=1.0, domain='positive')
        .parameter('eps', default=0.05, domain='positive')
        .parameter('D', default=0.5, domain='positive')
        .set_action_text('xt*(Dt*x + mu*x + eps*x^3) - D*xt^2')
        .equation(lhs='mu*x', rhs='-eps*x^3')
        .build())
kw = dict(k=2, max_ell=1, external_fields=[('x', 1), ('x', 1)], tau_grid=np.linspace(-4, 4, 17),
          parallel=False, verbose=False)
a = compute_cumulants(hand, **kw)
b = compute_cumulants(sde.build(), **kw)
print('max difference, tree and one loop:',
      [float(np.max(np.abs(np.asarray(a['C_tau_by_ell'][l]) - np.asarray(b['C_tau_by_ell'][l])))) for l in (0, 1)])

max difference, tree and one loop: [0.0, 0.0]


## 3. Extending an SDE model by hand

`SDE.extra_action(text)` appends a term to the derived action, and `sde.to_builder()` hands over a `TemporalModelBuilder` to continue with. For instance, a weak non-Gaussian correction to the noise written directly as a cumulant term ($-\kappa_3\tilde x^3/6$):

In [5]:
ext = (SDE('with a third cumulant')
       .parameter('mu', default=1.0, domain='positive').parameter('D', default=0.5, domain='positive')
       .parameter('k3', default=0.2)
       .field('x').drift('x', '-mu*x')
       .noise('xi', Gaussian(var='2*D'), couples={'x': 1})
       .extra_action('-k3*xt^3/6'))
ext.show()
r = compute_cumulants(ext.build(), k=3, max_ell=0, external_fields=[('x', 1)] * 3, tau_grid=np.array([0.0, 1.0]),
                      parallel=False, verbose=False)
print('equal-time third cumulant:', complex(r['total_C_by_ell'][0](0, 0, 0)).real, ' exact k3/(3 mu) =', 0.2 / 3)

# SDE 'with a third cumulant' (ito)
.set_action_text('''
sum(
    xt[i]*(Dt*x[i] - (-mu*x[i]))
    - (D*xt[i]^2)
    for i in pop)
+ (-k3*xt^3/6)
''')
.equation(lhs='Dt*x[i] + mu*x[i]', rhs='0', population='pop')
# note: mean-field equation of 'x': linear term -mu*x[i] moved to the lhs.
equal-time third cumulant: 0.06666666666666667  exact k3/(3 mu) = 0.06666666666666667


## 4. Model files, the config front end and the GUI

Models can be saved as `models/<name>.model.py` files and loaded with `dd.load_model`. `dd.Config` collects every run option (`dd.config_options()` lists them) and `dd.run` dispatches temporal and spatial models alike:

```python
model, mod = dd.load_model('ou_quartic')
cfg = dd.Config(k=2, max_ell=2, external_fields=[('dx', 1), ('dx', 1)], tau_grid=(-8, 8, 33), parallel=False)
res = dd.run(model, cfg, mod)
dd.plot_cumulant(res, cfg, model)
```

[`model_builder.ipynb`](../model_builder.ipynb) is a form-based editor (`ModelUI`) that writes such files, and [`model_runner.ipynb`](../model_runner.ipynb) runs any of them from one configuration cell. The previous guided tour of the hand-written builder is kept in [`legacy/`](../legacy/).

In [6]:
model, mod = dd.load_model('ou_quartic')
cfg = dd.Config(k=2, max_ell=1, external_fields=[('dx', 1), ('dx', 1)], tau_grid=(-6.0, 6.0, 25), parallel=False)
res = dd.run(model, cfg, mod)
print(dd.summary(res))

model : 'OU Quartic (white noise)'
k      : 2    max_ell : 1
fields : ['x']   spatial_dim : 0
